# 05 · Document Parsing：结构化解析

> 加载是“读进来”，解析是“读懂结构”。尤其 PDF 不能只做 `pdf→text`——那样表格、图片、层级全丢了。

**本文件覆盖知识点**：文本提取 / 表格提取 / 图片提取 / 标题识别 / 章节识别 / 页码 / Header-Footer / 文档结构恢复 / OCR / Layout Analysis

目标态：
```text
PDF ──> 文本 + 表格 + 图片 + 版面结构(标题/章节/页码)   ← 而不是纯 text
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. 为什么要“结构化”解析

同一份 PDF，两种解析结果：

```text
差:  "价格表¥998¥1288产品名基础版专业版"      # 表格横竖错位，读不出字段
好:  [表格] 产品名=基础版 价格=¥998            # 字段可检索、可过滤
    [图片] 架构图.png（单独存，供多模态检索）
    [标题] 第2节 价格体系（可用于层级切分/元数据）
```

保留结构，后面 **切分（07）**、**元数据（09）**、**多模态检索（30）** 才有料可用。

In [ ]:
# 知识点·真调说明：结构化解析的价值 —— 同一定价信息，“纯文本串”答不准，“结构化表格”能逐行作答
print('① 纯文本抽取：表格被压成几串文字，行列对应关系全丢')
_llm_live(
    prompt='某价格页经“纯文本抽取”(未做表格识别)后，各单元格被挤成三串，原行列关系已丢失：\n'
           '串1：998元/月  1999元/月  免费试用\n'
           '串2：基础版  专业版  企业版\n'
           '串3：公有云  支持私有化  定制\n'
           '（三串之间是否一一对应、对应顺序如何，抽取时已不可考）\n'
           '【问题】专业版一个月多少钱？它支持私有化部署吗？',
    system='你是售前顾问，只能依据上面原文判断；对应关系在原文里丢了就如实说明“无法确定”，不要强行配对猜测。',
    fallback='纯文本无法可靠作答：998/1999 与 基础/专业/企业 之间的一一对应在抽取时已丢失，'
             '可能 998 是专业版也可能 1999 才是；串3 也看不出“支持私有化”是哪个档位的能力——只能答“无法确定”。',
    temperature=0.2,
)
print()
print('② 结构化解析：同一信息还原成带行列的表，能定位、能逐行引用')
_llm_live(
    prompt='【结构化表格】\n'
           '| 档位 | 价格 | 部署方式 |\n'
           '| 基础版 | 998 元/月 | 公有云 |\n'
           '| 专业版 | 1999 元/月 | 公有云 + 支持私有化 |\n'
           '| 企业版 | 免费试用 | 定制部署 |\n'
           '【问题】专业版一个月多少钱？它支持私有化部署吗？请注明依据的是第几行。',
    system='你是售前顾问，按表格逐行作答，并注明依据行号。',
    fallback='专业版 1999 元/月，部署方式为“公有云 + 支持私有化”——依据表格第 2 行（专业版）。',
    temperature=0.2,
)
print()
print('同样的信息：纯文本只能靠猜/答“无法确定”，结构化表格则能确定并“按行引用”。')
print('→ 这就是 Parsing 要保留表格结构的原因：下游的检索与问答，需要的是“一一对应的字段”而不是一串无从拆分的字。')

In [ ]:
# 真实解析一份 PDF：页数 / 每页文本 / 每页图片，并给出「结构化页对象」的形状。
# 原来这里只打印「函数已就绪、把 PDF 放进 data/ 再说」——解析是纯本地计算、不依赖任何 Key，
# data/ 里既然已有真实样例，就该真读出内容；占位分支只在文件确实缺失时提示，且给出真实期望路径。
import fitz  # PyMuPDF
import csv, json
from pathlib import Path
from html.parser import HTMLParser
from collections import Counter

def pdf_to_structured_pages(pdf_path):
    """逐页解析 → 结构化页对象；text 之外保留版面块/图片等结构信息（第 2 节「金字塔」的底层）"""
    pages = []
    with fitz.open(pdf_path) as doc:
        for pno in range(len(doc)):
            page = doc[pno]
            text = page.get_text()
            # get_images(full=True) 给出每张图的 xref/宽高/色彩空间，可直接另存供多模态检索（第 30 课）
            imgs = [{'xref': x[0], '宽': x[2], '高': x[3], '色彩空间': x[5]}
                    for x in page.get_images(full=True)]
            pages.append({'page': pno + 1, 'text': text, 'chars': len(text.strip()),
                          'blocks': len(page.get_text('blocks')), 'images': imgs,
                          'title': text.strip().splitlines()[0] if text.strip() else ''})
    return pages

PDF = Path('data/星云产品手册.pdf')
if not PDF.exists():
    print('未找到 %s —— 请把待解析的 PDF 放到 %s 后重跑本 cell。' % (PDF, PDF.parent.resolve()))
else:
    pages = pdf_to_structured_pages(PDF)
    print('解析 %s：共 %d 页' % (PDF.name, len(pages)))
    for p in pages:
        print('  第%d页 | %3d字 | 版面块%2d | 图片%d张 | 首行: %s'
              % (p['page'], p['chars'], p['blocks'], len(p['images']), p['title']))
        print('         正文: %s…' % p['text'].strip().replace('\n', ' ')[:76])
    print('\n结构化页对象长这样（第 1 页，text 截断展示）:')
    _p0 = dict(pages[0]); _p0['text'] = _p0['text'][:34] + '…'
    print(json.dumps(_p0, ensure_ascii=False, indent=2))
    print('\n各页图片（真实提取的元信息，另存后即可交给多模态检索）:')
    for p in pages:
        for im in p['images']:
            print('  第%d页 xref=%d %dx%d %s' % (p['page'], im['xref'], im['宽'], im['高'], im['色彩空间']))

print('\n' + '=' * 62)
print('多格式加载：同属一个业务知识库的样例，各自走对应解析器，产出可入库的统一结构')

# ① CSV：表格 → 行对象（字段可过滤、可聚合，正是「结构化 vs 一串文字」的区别）
with open('data/样例工单.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print('\n① CSV《样例工单.csv》：%d 行 × %d 列' % (len(rows), len(rows[0])))
print('   列: %s' % ' / '.join(rows[0].keys()))
print('   首行: %s | %s | %s | 满意度 %s'
      % (rows[0]['工单号'], rows[0]['问题类型'], rows[0]['渠道'], rows[0]['满意度']))
print('   按问题类型聚合: %s' % dict(Counter(r['问题类型'] for r in rows)))
print('   转人工比例: %d/%d' % (sum(r['是否转人工'] == '是' for r in rows), len(rows)))

# ② JSON：知识库元数据 → 文档清单（status=failed 的文档连失败原因一起留下）
kb = json.loads(Path('data/样例知识库元数据.json').read_text(encoding='utf-8'))
meta = kb['knowledge_base']
print('\n② JSON《样例知识库元数据.json》')
print('   知识库: %s (%s) 向量模型=%s 维度=%d 切片=%d/overlap%d'
      % (meta['name'], meta['kb_id'], meta['embedding_model'], meta['embedding_dim'],
         meta['chunking']['chunk_size'], meta['chunking']['overlap']))
for d in kb['documents']:
    print('   - %s %-8s %-6s %s' % (d['doc_id'], d['format'], d['status'], d.get('error', '')))

# ③ HTML：从标签里还原标题/正文/表格（标准库即可；生产可直接换 Unstructured / Docling）
class _HelpCenterParser(HTMLParser):
    """抽 title / h1-h3 / 段落 / 表格行 —— 轻量但真实；表格按 <tr> 聚行，行列关系不丢"""
    def __init__(self):
        super().__init__()
        self.stack = []; self.title = ''; self.heads = []; self.texts = []
        self.rows = []; self._row = None
    def handle_starttag(self, tag, attrs):
        self.stack.append(tag)
        if tag == 'tr':
            self._row = []
    def handle_endtag(self, tag):
        if tag == 'tr' and self._row is not None:
            self.rows.append(self._row); self._row = None
        if tag in self.stack:
            self.stack = self.stack[:self.stack.index(tag)]
    def handle_data(self, data):
        s = data.strip()
        if not s:
            return
        cur = self.stack[-1] if self.stack else ''
        if cur == 'title':
            self.title = s
        elif cur in ('h1', 'h2', 'h3'):
            self.heads.append(s)
        elif cur in ('p', 'li'):
            self.texts.append(s)
        elif cur in ('td', 'th') and self._row is not None:
            self._row.append(s)

hp = _HelpCenterParser()
hp.feed(Path('data/样例帮助中心.html').read_text(encoding='utf-8'))
print('\n③ HTML《样例帮助中心.html》')
print('   title: %s' % hp.title)
print('   小节标题: %s' % ' / '.join(hp.heads))
print('   正文段落 %d 条，首段: %s' % (len(hp.texts), hp.texts[0][:46]))
print('   表格 %d 行；表头 %s' % (len(hp.rows), hp.rows[0]))
print('   末行（选中 chunk_size 这一行）: %s' % hp.rows[-1])

# 解析不是终点：产出要能进同一套索引，才算「解析完成」
print('\n' + '=' * 62)
print('底座当前索引的语料：%d 篇 → %d 个片段' % (len({c['source'] for c in CHUNKS}), len(CHUNKS)))
print('  来源: %s' % '、'.join(sorted({c['source'] for c in CHUNKS})))
print('→ PDF/CSV/JSON/HTML 解析出的内容，要整理成同样的 {text, source, section} 片段结构，'
      '才能和这批 Markdown 一起进向量 / BM25 索引；本课的解析只负责把结构「读出来」。')


## 2. 解析金字塔（自底向上）

| 层次 | 提取内容 | 产出 |
|------|---------|------|
| 版面(Layout) | 文字块/图片块坐标 | Layout Analysis：分栏还原阅读顺序 |
| 结构 | 标题层级/章节/页码 | 供层级切分与元数据 |
| 对象 | 表格(行列表头)、图片、公式 | 表格结构化、图片另存 |
| 像素 | 扫描件文字 | OCR 识别成文本再走上面流程 |

常见工具：
- 轻量：**PyMuPDF**(block/坐标)、**PDFPlumber**(表格定位)；
- 生产：**Unstructured**(partition_pdf)、**Docling**(版面+表格)、**MinerU**(中文/数学公式强)；
- OCR：**PaddleOCR** / **RapidOCR**(中文好)、Tesseract(通用)。

## 3. OCR：扫描件怎么进 RAG

```text
扫描图片 ──> OCR(版面检测+文字识别) ──> 带坐标的文本 ──> 表格还原/结构解析 ──> 向量库
```

OCR 的常见坑：
- 表格会被 OCR 成“无空格”的文字，需要**版面模型**还原行列；
- 双栏文档需要按栏还原顺序，否则正文前后颠倒；
- 繁体/竖排/公式需要专门的模型。



In [ ]:
# 知识点·真调说明：OCR 后处理 —— 扫描件转出的字带错，交给模型按上下文做“词级纠错”
_llm_live(
    prompt='下面是某扫描件经 OCR 后的原文，存在空格被吞、形近字写错、断句错位等问题：\n'
           '【OCR原文】本产 品支寺公有云SaaS 与私 有化部暑，答不上来的问题会自动转接人 工，并携带完 整的对话上下文。\n'
           '请把它还原成规范的一句话：只修 OCR 引入的错字/断句，不要增删或改写原文意思；'
           '第二行用“<错字→改正>”列表说明你改了哪几处。',
    system='你是 OCR 后处理助手，只做识别纠错与断句，不添加原文没有的信息。输出两行：纠错后的句子；<错字→改正>列表。',
    fallback='纠错后：本产品支持公有云 SaaS 与私有化部署，答不上来的问题会自动转接人工，并携带完整的对话上下文。\n'
             '<支寺→支持>  <部暑→部署>  <转接人 工→转接人工>  <完 整→完整>',
    temperature=0.2,
)
print('→ OCR 把“像素”变成“文字”，但常夹带错字与粘连；让 LLM 按上下文做词级纠错是常见的 OCR 后处理手段——'
      '再往前才是 06 课的规则化清洗（控制字符 / 零宽 / 空白规整），两道关卡一起保证进库文本干净。')

## 小结

- Parsing 的目标是 **文本+表格+图片+结构** 四件套；
- 按复杂度选工具：轻量 PyMuPDF / 生产 Unstructured·Docling·MinerU；
- 扫描件先 OCR。

解析出的长文本仍不能直接向量化：先做**数据清洗（06）**，再把它切成 chunk（07）。